# CogVideoX-5B I2V on Kaggle (Dual GPU)

## Setup
- Settings → Accelerator → **GPU T4 x2**
- Settings → Internet → **On**

## Upload template image
- Right panel **+ Add Input** → **Upload**
- Copy file path after upload

## 1. Install

In [ ]:
!pip install -q diffusers transformers accelerate torchvision opencv-python pillow

## 2. Check GPUs

In [ ]:
!nvidia-smi

## 3. Settings

In [ ]:
IMAGE_PATH = "/kaggle/input/YOUR_DATASET/character_template.png"
OUTPUT_DIR = "/kaggle/working/"
PROMPT = "a cute anime mother holding her baby, gentle horizontal sway, warm pastel colors, simple background"
NEGATIVE_PROMPT = "ugly, blurry, deformed, text, watermark, complex background, realistic"
SEED = 42
NUM_FRAMES = 25
NUM_STEPS = 30
GUIDANCE_SCALE = 7.0
WIDTH = 480
HEIGHT = 720

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)
print("OK")

## 4. Load model

In [ ]:
import torch
from diffusers import CogVideoXImageToVideoPipeline
from diffusers.utils import export_to_video
from PIL import Image
import time

print("Loading model (first run downloads ~10GB)...")
t0 = time.time()

pipe = CogVideoXImageToVideoPipeline.from_pretrained(
    "THUDM/CogVideoX-5b-I2V",
    torch_dtype=torch.bfloat16,
    device_map="balanced"
)

pipe.enable_attention_slicing()
pipe.vae.enable_tiling()

print(f"Model loaded in {time.time()-t0:.1f}s")
print(f"GPUs: {torch.cuda.device_count()}")

## 5. Run inference

In [ ]:
print(f"Loading image: {IMAGE_PATH}")
image = Image.open(IMAGE_PATH).convert("RGB")
print(f"Original size: {image.size}")

image = image.resize((WIDTH, HEIGHT), Image.LANCZOS)
print(f"Resized to: {image.size}")

print(f"\nGenerating {NUM_FRAMES} frames @ {WIDTH}x{HEIGHT} {NUM_STEPS} steps...")
t0 = time.time()

generator = torch.Generator(device="cpu").manual_seed(SEED)

with torch.inference_mode():
    video_frames = pipe(
        prompt=PROMPT,
        image=image,
        num_frames=NUM_FRAMES,
        guidance_scale=GUIDANCE_SCALE,
        num_inference_steps=NUM_STEPS,
        generator=generator,
        negative_prompt=NEGATIVE_PROMPT,
    ).frames[0]

elapsed = time.time() - t0
print(f"Done in {elapsed:.1f}s ({len(video_frames)} frames)")

## 6. Save & preview

In [ ]:
output_file = os.path.join(OUTPUT_DIR, "cogvideo_output.mp4")
export_to_video(video_frames, output_file, fps=8)
print(f"Saved: {output_file}")
print(f"Size: {os.path.getsize(output_file)/1024/1024:.1f} MB")

from IPython.display import Video, display
display(Video(output_file, width=480))